# Prova: os dados vêm do banco

Este notebook não lê nenhum CSV. Ele abre só `dados/processed/tse.duckdb` em modo leitura e mostra que:

1. o schema `modelo` tem tabelas de verdade (não views sobre CSV);
2. as tabelas têm linhas gravadas no arquivo;
3. o plano de execução de uma consulta lê a tabela (`SEQ_SCAN`), não um arquivo (`READ_CSV`);
4. as chaves estrangeiras do modelo ligam as tabelas.

Pré-requisito: `uv run scripts/carregar_staging.py` já rodado.

In [1]:
from pathlib import Path

import duckdb

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
DB = ROOT / "dados" / "processed" / "tse.duckdb"

# read_only: o notebook não consegue alterar o banco, só consultar
con = duckdb.connect(str(DB), read_only=True)
con.sql("SELECT database_name, path, readonly FROM duckdb_databases() WHERE NOT internal").df()

,database_name,path,readonly
0,tse,/home/davi/Documentos/UFPI/6-periodo/BDR/anali...,True


## 1. O que existe no schema `modelo`

`table_type = BASE TABLE` significa tabela gravada; uma view apareceria como `VIEW`.

In [2]:
con.sql("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'modelo'
    ORDER BY table_name
""").df()

,table_name,table_type
0,agente_financeiro,BASE TABLE
1,bem_candidato,BASE TABLE
2,candidatura,BASE TABLE
3,cargo,BASE TABLE
4,censo_faixa_etaria,BASE TABLE
5,censo_instrucao,BASE TABLE
6,comparecimento_municipio,BASE TABLE
7,comparecimento_perfil,BASE TABLE
8,despesa_campanha,BASE TABLE
9,eleicao,BASE TABLE


## 2. Quantas linhas cada tabela guarda

In [3]:
tabelas = [
    t for (t,) in con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'modelo' ORDER BY 1").fetchall()
]
con.sql(
    " UNION ALL ".join(f"SELECT '{t}' AS tabela, count(*) AS linhas FROM modelo.{t}" for t in tabelas)
    + " ORDER BY linhas DESC"
).df()

,tabela,linhas
0,votacao_candidato_municipio,17274440
1,bem_candidato,3156763
2,candidatura,2949269
3,politico,1855674
4,votacao_legenda_municipio,1257564
5,despesa_campanha,364147
6,receita_campanha,171252
7,censo_faixa_etaria,116970
8,comparecimento_municipio,106120
9,agente_financeiro,105260


## 3. O plano de execução lê a tabela, não o CSV

Se a consulta fosse sobre uma view de CSV, o plano mostraria `READ_CSV`.

In [4]:
plano = con.sql(
    "EXPLAIN SELECT ano, sum(vr_receita) FROM modelo.receita_campanha r "
    "JOIN modelo.candidatura c USING (id_candidatura) GROUP BY ano"
).fetchall()[0][1]
print(plano)
print("lê CSV?", "READ_CSV" in plano.upper())

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_integ│
│   ral_integer(#0, 2002)   │
│             #1            │
│                           │
│        ~3,057 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│   PERFECT_HASH_GROUP_BY   │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: sum(#1)    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│            ano            │
│         vr_receita        │
│                           │
│       ~227,880 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_integra│
│    l_utinyint(#0, 2002)   │
│             #1            │
│                           │
│       ~227,880 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         

## 4. Uma consulta que só funciona porque as tabelas estão ligadas

Os 10 candidatos do PI que mais receberam, juntando `receita_campanha → candidatura → politico / cargo`.

In [5]:
con.sql("""
    SELECT c.ano, p.nm_candidato, g.ds_cargo, c.sg_ue,
           round(sum(r.vr_receita), 2) AS total_recebido
    FROM modelo.receita_campanha r
    JOIN modelo.candidatura c USING (id_candidatura)
    JOIN modelo.politico   p USING (nr_titulo_eleitoral)
    JOIN modelo.cargo      g USING (cd_cargo)
    GROUP BY ALL
    ORDER BY total_recebido DESC
    LIMIT 10
""").df()

,ano,nm_candidato,ds_cargo,sg_ue,total_recebido
0,2022,SILVIO MENDES DE OLIVEIRA FILHO,GOVERNADOR,PI,7261342.14
1,2026,JOEL RODRIGUES DA SILVA,GOVERNADOR,PI,7154680.60
2,2014,ANTONIO JOSÉ DE MORAES SOUZA FILHO,GOVERNADOR,PI,6772331.07
3,2014,IRACEMA MARIA PORTELLA NUNES NOGUEIRA LIMA,DEPUTADO FEDERAL,PI,6723539.50
4,2022,RAFAEL TAJRA FONTELES,GOVERNADOR,PI,5974482.91
5,2014,JOSE WELLINGTON BARROSO DE ARAUJO DIAS,GOVERNADOR,PI,4870380.03
6,2014,HERÁCLITO DE SOUSA FORTES,DEPUTADO FEDERAL,PI,4136768.12
7,2022,JOEL RODRIGUES DA SILVA,SENADOR,PI,4024654.87
8,2026,TIAGO MAXIMIANO JUNQUEIRA,SENADOR,PI,3836500.00
9,2026,CIRO NOGUEIRA LIMA FILHO,SENADOR,PI,3818280.53


In [6]:
con.close()